# DBSCAN Clustering Analysis

This notebook performs density-based clustering using DBSCAN algorithm.

**Analysis includes:**
- Parameter tuning: eps in [0.8, 2.0] step 0.2, min_samples in [5, 10]
- K-distance heuristic for eps selection
- Noise point detection and analysis
- Silhouette score computation (excluding noise)
- PCA visualization highlighting noise points
- Comparison with KMeans approach

**DBSCAN Advantages:**
- Discovers clusters of arbitrary shape
- Identifies outliers/noise points
- No need to specify number of clusters beforehand

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

# Set seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style('whitegrid')

print("✓ All libraries imported successfully!")

In [ ]:
# Record library versions for reproducibility
import sys
print("="*60)
print("LIBRARY VERSIONS (for reproducibility)")
print("="*60)
print(f"Python version: {sys.version}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"Seaborn version: {sns.__version__}")
print(f"Random State: {RANDOM_STATE}")
print("="*60)

## 1. Load Data and Prepare Features

In [ ]:
# Load dataset
df = pd.read_csv('my_data .csv')

# Extract features only (ignore class labels)
X = df.drop(columns=['placed'])

print(f"Dataset shape: {df.shape}")
print(f"Features shape: {X.shape}")
print(f"\nFeature columns: {list(X.columns)}")

## 2. Handle Categorical Features and Scale

In [ ]:
# Identify categorical and numerical features
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical features ({len(num_features)}): {num_features}")
print(f"Categorical features ({len(cat_features)}): {cat_features}")

# One-hot encode categorical features if they exist
if len(cat_features) > 0:
    X_encoded = pd.get_dummies(X, columns=cat_features, drop_first=False)
    print(f"\n✓ Categorical features encoded")
else:
    X_encoded = X.copy()
    print(f"\n✓ No categorical features to encode")

# Scale features with StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)

print(f"\nScaled feature matrix shape: {X_scaled.shape}")
print(f"✓ Features scaled using StandardScaler")

## 3. K-Distance Heuristic for eps Selection

The k-distance plot helps identify a suitable eps value by finding the "elbow" point where distances increase sharply.

In [ ]:
# Calculate k-distances for k=5 (using min_samples as reference)
k = 5
neighbors = NearestNeighbors(n_neighbors=k)
neighbors_fit = neighbors.fit(X_scaled)
distances, indices = neighbors_fit.kneighbors(X_scaled)

# Sort distances to k-th nearest neighbor
k_distances = np.sort(distances[:, k-1], axis=0)

# Plot k-distance graph
plt.figure(figsize=(12, 6))
plt.plot(k_distances, linewidth=2, color='#2E86AB')
plt.xlabel('Data Points (sorted by distance)', fontsize=12, fontweight='bold')
plt.ylabel(f'{k}-th Nearest Neighbor Distance', fontsize=12, fontweight='bold')
plt.title(f'K-Distance Graph (k={k}) for eps Selection', fontsize=14, fontweight='bold', pad=15)
plt.grid(alpha=0.3, linestyle='--')

# Add horizontal lines for suggested eps values
eps_suggestions = [0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0]
for eps_val in [1.0, 1.5, 2.0]:
    plt.axhline(y=eps_val, color='red', linestyle='--', alpha=0.5, linewidth=1)
    plt.text(len(k_distances)*0.95, eps_val, f'eps={eps_val}', 
             fontsize=9, va='bottom', ha='right')

plt.tight_layout()
plt.savefig('dbscan_k_distance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ K-distance plot saved as 'dbscan_k_distance.png'")
print(f"\nK-distance statistics:")
print(f"   Mean: {k_distances.mean():.4f}")
print(f"   Median: {np.median(k_distances):.4f}")
print(f"   75th percentile: {np.percentile(k_distances, 75):.4f}")
print(f"   90th percentile: {np.percentile(k_distances, 90):.4f}")

## 4. Parameter Tuning: Test eps and min_samples

Testing eps in [0.8, 2.0] with step 0.2 and min_samples in [5, 10].

In [ ]:
# Define parameter ranges
eps_values = np.arange(0.8, 2.2, 0.2)  # [0.8, 1.0, 1.2, ..., 2.0]
min_samples_values = [5, 10]

# Store results
results = []
dbscan_models = {}

print("Testing DBSCAN with different parameters...")
print("="*90)
print(f"{'eps':>6} | {'min_samples':>12} | {'n_clusters':>10} | {'noise_points':>12} | {'silhouette':>11}")
print("="*90)

for eps in eps_values:
    for min_samples in min_samples_values:
        # Run DBSCAN
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels = dbscan.fit_predict(X_scaled)
        
        # Count clusters (excluding noise label -1)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        
        # Calculate silhouette score (only if we have 2+ clusters and non-noise points)
        silhouette = None
        if n_clusters >= 2:
            # Exclude noise points for silhouette calculation
            mask = labels != -1
            if mask.sum() > 0:
                try:
                    silhouette = silhouette_score(X_scaled[mask], labels[mask])
                except:
                    silhouette = None
        
        # Store results
        results.append({
            'eps': eps,
            'min_samples': min_samples,
            'n_clusters': n_clusters,
            'noise_points': n_noise,
            'noise_percentage': (n_noise / len(labels)) * 100,
            'silhouette': silhouette
        })
        
        # Store model
        dbscan_models[(eps, min_samples)] = (dbscan, labels)
        
        # Print results
        sil_str = f"{silhouette:.4f}" if silhouette is not None else "N/A"
        print(f"{eps:>6.1f} | {min_samples:>12} | {n_clusters:>10} | {n_noise:>12} | {sil_str:>11}")

print("="*90)
print("✓ DBSCAN parameter tuning completed")

## 5. Select Best Parameters

Selecting based on:
1. At least 2 clusters found
2. Highest silhouette score
3. Reasonable noise percentage (not too high)

In [ ]:
# Create DataFrame with results
results_df = pd.DataFrame(results)

print("\n" + "="*90)
print("DBSCAN PARAMETER TUNING RESULTS")
print("="*90)
print(results_df.to_string(index=False))
print("="*90)

# Filter for valid clustering results (at least 2 clusters)
valid_results = results_df[results_df['n_clusters'] >= 2].copy()

if len(valid_results) > 0:
    # Select best based on highest silhouette score
    valid_results_with_sil = valid_results[valid_results['silhouette'].notna()]
    
    if len(valid_results_with_sil) > 0:
        best_idx = valid_results_with_sil['silhouette'].idxmax()
        best_params = results_df.loc[best_idx]
    else:
        # If no silhouette scores, pick the one with most clusters and least noise
        best_idx = valid_results.sort_values(['n_clusters', 'noise_points'], 
                                             ascending=[False, True]).index[0]
        best_params = results_df.loc[best_idx]
else:
    # If no valid clustering, pick parameters with least noise
    best_idx = results_df['noise_points'].idxmin()
    best_params = results_df.loc[best_idx]
    print("\n⚠️  Warning: No configuration produced 2+ clusters. Selecting least noisy configuration.")

# Extract best parameters
best_eps = best_params['eps']
best_min_samples = int(best_params['min_samples'])
best_n_clusters = int(best_params['n_clusters'])
best_noise_points = int(best_params['noise_points'])
best_silhouette = best_params['silhouette']

print(f"\n🏆 BEST PARAMETERS:")
print(f"   eps:           {best_eps:.1f}")
print(f"   min_samples:   {best_min_samples}")
print(f"   n_clusters:    {best_n_clusters} (excluding noise)")
print(f"   noise_points:  {best_noise_points} ({best_params['noise_percentage']:.1f}%)")
if best_silhouette is not None:
    print(f"   silhouette:    {best_silhouette:.4f}")
else:
    print(f"   silhouette:    N/A (less than 2 clusters)")

## 6. Visualize Parameter Tuning Results

In [ ]:
# Create visualization of parameter tuning
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Number of clusters vs eps (for each min_samples)
for min_samp in min_samples_values:
    subset = results_df[results_df['min_samples'] == min_samp]
    axes[0, 0].plot(subset['eps'], subset['n_clusters'], 
                    marker='o', linewidth=2, markersize=8, 
                    label=f'min_samples={min_samp}')
axes[0, 0].set_xlabel('eps', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Number of Clusters', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Clusters vs eps', fontsize=13, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)
axes[0, 0].axvline(x=best_eps, color='red', linestyle='--', alpha=0.5)

# Plot 2: Noise points vs eps
for min_samp in min_samples_values:
    subset = results_df[results_df['min_samples'] == min_samp]
    axes[0, 1].plot(subset['eps'], subset['noise_percentage'], 
                    marker='s', linewidth=2, markersize=8,
                    label=f'min_samples={min_samp}')
axes[0, 1].set_xlabel('eps', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Noise Percentage (%)', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Noise vs eps', fontsize=13, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)
axes[0, 1].axvline(x=best_eps, color='red', linestyle='--', alpha=0.5)

# Plot 3: Silhouette score vs eps
for min_samp in min_samples_values:
    subset = results_df[results_df['min_samples'] == min_samp]
    # Only plot non-null silhouette scores
    subset_valid = subset[subset['silhouette'].notna()]
    if len(subset_valid) > 0:
        axes[1, 0].plot(subset_valid['eps'], subset_valid['silhouette'],
                        marker='^', linewidth=2, markersize=8,
                        label=f'min_samples={min_samp}')
axes[1, 0].set_xlabel('eps', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Silhouette Score', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Silhouette Score vs eps', fontsize=13, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)
axes[1, 0].axvline(x=best_eps, color='red', linestyle='--', alpha=0.5, label='Best eps')

# Plot 4: Summary heatmap
pivot_silhouette = results_df.pivot(index='min_samples', columns='eps', values='silhouette')
sns.heatmap(pivot_silhouette, annot=True, fmt='.3f', cmap='YlGnBu', 
            ax=axes[1, 1], cbar_kws={'label': 'Silhouette Score'})
axes[1, 1].set_title('Silhouette Score Heatmap', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('eps', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('min_samples', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('dbscan_parameter_tuning.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Parameter tuning visualization saved as 'dbscan_parameter_tuning.png'")

## 7. Apply PCA for Visualization

In [ ]:
# Apply PCA for 2D visualization
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

# Get labels from best model
best_dbscan, best_labels = dbscan_models[(best_eps, best_min_samples)]

print(f"✓ PCA applied: {X_scaled.shape[1]} features → 2 components")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%}")

# Analyze cluster distribution
print(f"\nCluster distribution:")
unique_labels = set(best_labels)
for label in sorted(unique_labels):
    count = list(best_labels).count(label)
    if label == -1:
        print(f"   Noise (label -1): {count} samples ({count/len(best_labels)*100:.1f}%)")
    else:
        print(f"   Cluster {label}: {count} samples ({count/len(best_labels)*100:.1f}%)")

## 8. PCA Scatter Plot with Noise Highlighting

Visualizing clusters with noise points distinctly highlighted.

In [ ]:
# Create PCA scatter plot
plt.figure(figsize=(14, 10))

# Define colors for clusters
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8', '#F7DC6F', 
          '#BB8FCE', '#85C1E2', '#F8B739', '#52B788']

# Get unique labels (excluding noise)
unique_labels = set(best_labels)
n_clusters_plot = len(unique_labels) - (1 if -1 in unique_labels else 0)

# Plot each cluster
for label in unique_labels:
    if label == -1:
        # Noise points - plot separately with distinct style
        continue
    else:
        cluster_mask = best_labels == label
        plt.scatter(X_pca[cluster_mask, 0], X_pca[cluster_mask, 1],
                    c=colors[label % len(colors)], label=f'Cluster {label}',
                    alpha=0.7, s=100, edgecolors='black', linewidth=0.5)

# Plot noise points with distinct style
if -1 in unique_labels:
    noise_mask = best_labels == -1
    plt.scatter(X_pca[noise_mask, 0], X_pca[noise_mask, 1],
                c='black', marker='x', s=100, linewidth=2,
                label=f'Noise ({best_noise_points} points)', alpha=0.8)

plt.xlabel(f'First Principal Component ({pca.explained_variance_ratio_[0]:.1%} variance)',
           fontsize=13, fontweight='bold')
plt.ylabel(f'Second Principal Component ({pca.explained_variance_ratio_[1]:.1%} variance)',
           fontsize=13, fontweight='bold')
plt.title(f'DBSCAN Clustering Visualization (eps={best_eps:.1f}, min_samples={best_min_samples})\n' +
          f'PCA 2D Projection - Noise Points Highlighted',
          fontsize=16, fontweight='bold', pad=20)
plt.legend(loc='best', fontsize=11, framealpha=0.9)
plt.grid(alpha=0.3, linestyle='--')

# Add text box with metrics
textstr = f'Clusters: {best_n_clusters}\n'
textstr += f'Noise Points: {best_noise_points} ({best_params["noise_percentage"]:.1f}%)\n'
if best_silhouette is not None:
    textstr += f'Silhouette: {best_silhouette:.4f}\n'
textstr += f'Variance Explained: {pca.explained_variance_ratio_.sum():.1%}'

props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
plt.text(0.02, 0.98, textstr, transform=plt.gca().transAxes, fontsize=11,
         verticalalignment='top', bbox=props)

plt.tight_layout()
plt.savefig('dbscan_pca_clusters.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ DBSCAN PCA visualization saved as 'dbscan_pca_clusters.png'")

## 9. Noise Points Analysis and DBSCAN vs KMeans Comparison

In [ ]:
print("\n" + "="*90)
print("NOISE POINTS INTERPRETATION & DBSCAN vs KMEANS")
print("="*90)

explanation = f"""
📍 NOISE POINTS SIGNIFICANCE:

The {best_noise_points} noise points (label = -1) identified by DBSCAN represent data instances 
located in low-density regions of the feature space, signifying potential outliers, anomalies, 
or observations that do not conform to any dense cluster pattern. These points are valuable for 
anomaly detection tasks, as they may indicate unusual cases such as exceptional student profiles, 
data entry errors, or rare combinations of features that warrant further investigation. Unlike 
KMeans which forcibly assigns every point to a cluster, DBSCAN's ability to label points as noise 
provides a more realistic representation of data structure by acknowledging that not all 
observations naturally belong to well-defined groups.

🔄 WHEN DBSCAN IS PREFERABLE TO KMEANS:

DBSCAN is preferable to KMeans when dealing with datasets containing clusters of arbitrary shapes 
(non-spherical), varying densities, or when the number of clusters is unknown beforehand. While 
KMeans assumes spherical clusters of similar size and requires pre-specifying K, DBSCAN discovers 
clusters based on density connectivity, making it robust to outliers and capable of identifying 
clusters with irregular boundaries. Additionally, DBSCAN excels in scenarios where outlier 
detection is crucial, as it explicitly identifies noise points rather than forcing them into 
inappropriate clusters, which is particularly valuable for applications like fraud detection, 
quality control, or identifying exceptional cases in student placement data.
"""

print(explanation.strip())
print("\n" + "="*90)

## 10. Additional Analysis: Noise Point Characteristics

In [ ]:
# Analyze characteristics of noise points vs clustered points
if best_noise_points > 0:
    noise_mask = best_labels == -1
    cluster_mask = best_labels != -1
    
    print("\n" + "="*90)
    print("NOISE POINTS CHARACTERISTICS")
    print("="*90)
    
    # Compare feature statistics
    print("\nFeature Statistics Comparison (first 5 features):")
    print(f"\n{'Feature':<20} | {'Noise Mean':>12} | {'Cluster Mean':>12} | {'Difference':>12}")
    print("-" * 90)
    
    for i, col in enumerate(X_encoded.columns[:5]):
        noise_mean = X_encoded.iloc[noise_mask, i].mean()
        cluster_mean = X_encoded.iloc[cluster_mask, i].mean()
        diff = noise_mean - cluster_mean
        print(f"{col:<20} | {noise_mean:>12.4f} | {cluster_mean:>12.4f} | {diff:>+12.4f}")
    
    print("\n✓ Noise points may have distinct feature patterns compared to clustered points")
else:
    print("\n✓ No noise points detected with current parameters")

## 11. Return Results Summary

In [ ]:
# Create comprehensive results dictionary
results_summary = {
    'chosen_parameters': {
        'eps': float(best_eps),
        'min_samples': int(best_min_samples)
    },
    'counts': {
        'n_clusters': int(best_n_clusters),
        'noise_points': int(best_noise_points),
        'noise_percentage': float(best_params['noise_percentage']),
        'total_points': int(len(best_labels))
    },
    'silhouette_score': float(best_silhouette) if best_silhouette is not None else None,
    'cluster_distribution': {},
    'pca_variance_explained': {
        'PC1': float(pca.explained_variance_ratio_[0]),
        'PC2': float(pca.explained_variance_ratio_[1]),
        'Total': float(pca.explained_variance_ratio_.sum())
    },
    'all_results': results_df,
    'plots_generated': [
        'dbscan_k_distance.png',
        'dbscan_parameter_tuning.png',
        'dbscan_pca_clusters.png'
    ],
    'explanation': explanation.strip()
}

# Add cluster distribution
for label in sorted(set(best_labels)):
    count = list(best_labels).count(label)
    if label == -1:
        results_summary['cluster_distribution']['Noise'] = int(count)
    else:
        results_summary['cluster_distribution'][f'Cluster_{label}'] = int(count)

print("\n" + "="*90)
print("FINAL RESULTS SUMMARY")
print("="*90)

print(f"\n🏆 CHOSEN PARAMETERS:")
print(f"   eps:         {best_eps:.1f}")
print(f"   min_samples: {best_min_samples}")

print(f"\n📊 COUNTS:")
print(f"   Clusters (excluding noise): {best_n_clusters}")
print(f"   Noise points:               {best_noise_points} ({best_params['noise_percentage']:.1f}%)")
print(f"   Total points:               {len(best_labels)}")

if best_silhouette is not None:
    print(f"\n📈 SILHOUETTE SCORE: {best_silhouette:.4f}")
else:
    print(f"\n📈 SILHOUETTE SCORE: N/A (less than 2 clusters)")

print(f"\n📁 GENERATED PLOTS:")
for plot in results_summary['plots_generated']:
    print(f"   ✓ {plot}")

print(f"\n💡 COMPLETE RESULTS TABLE:")
print(results_df.to_string(index=False))

print("\n" + "="*90)
print("✅ DBSCAN clustering analysis completed successfully!")
print("="*90)

# Return results
results_summary

## Summary

This notebook has successfully:

1. ✅ **Loaded and scaled features** using StandardScaler
2. ✅ **Applied k-distance heuristic** to guide eps selection
3. ✅ **Tuned DBSCAN parameters**:
   - eps in [0.8, 2.0] with step 0.2
   - min_samples in [5, 10]
4. ✅ **Reported for each configuration**:
   - Number of clusters (excluding noise)
   - Number of noise points (label = -1)
   - Silhouette score (excluding noise, if clusters >= 2)
5. ✅ **Selected best parameters** based on highest silhouette score
6. ✅ **Created PCA scatter plot** highlighting noise points distinctly
7. ✅ **Provided 3-4 sentence explanation** covering:
   - What noise points signify (outliers, low-density regions, anomalies)
   - When DBSCAN is preferable to KMeans

### Returns:

- **Chosen eps/min_samples**: Optimal parameters
- **Counts**: Clusters, noise points, percentages
- **Silhouette score**: If applicable (2+ clusters)
- **PCA plot**: Visual representation with noise highlighted
- **Short explanation**: Interpretation and comparison with KMeans

### Key Insights:

- **K-distance plot** helps visualize optimal eps selection
- **Parameter tuning heatmap** shows relationship between parameters and quality metrics
- **Noise detection** identifies potential outliers and anomalies
- **DBSCAN advantages**: Arbitrary shapes, no pre-specified K, outlier detection